# About

What we're trying to do is use master_units.parquet as our new source of truth, while also adding in things that we feel like are necessary in order to get what we need out of our apartments scrape. The first part is getting the right subset of apartments using the addresses, and then the second part is just doing a cron job with checkpoints to scrape all of them. 

In [10]:
import pandas as pd
import os
os.chdir('/home/kevinphan981/Programming/apartment-rental-scraper/')
df = pd.read_parquet('data/master_units.parquet', engine = 'pyarrow')
master_address = pd.read_csv('data/master_tmk_address.csv')
condo_listings = pd.read_csv('data/condo_listings.csv')
print(df.head(n = 10))
# print(master_address.head(n = 10))
print(condo_listings.head(n = 10))
print(df.shape) # about 6000, which is around what we're looking for.

     master_tmk parcel_number unit_number      modified_address
0  210020160000  210020160001        1401  66 QUEEN ST APT 1401
1  210020160000  210020160002        1402  66 QUEEN ST APT 1402
2  210020160000  210020160003        1405  66 QUEEN ST APT 1405
3  210020160000  210020160004        1501  66 QUEEN ST APT 1501
4  210020160000  210020160005        1502  66 QUEEN ST APT 1502
5  210020160000  210020160006        1503  66 QUEEN ST APT 1503
6  210020160000  210020160007        1504  66 QUEEN ST APT 1504
7  210020160000  210020160008        1505  66 QUEEN ST APT 1505
8  210020160000  210020160009        1601  66 QUEEN ST APT 1601
9  210020160000  210020160010        1602  66 QUEEN ST APT 1602
  listing_id              building_name              address unit_number  \
0    yfdelbs              Lilia Waikiki       2380 Kuhio Ave         NaN   
1    yfdelbs              Lilia Waikiki       2380 Kuhio Ave         NaN   
2    yfdelbs              Lilia Waikiki       2380 Kuhio Ave        

In [24]:
# let's try to join master_tmk_address.csv valid addresses (which we can only guess is right) with the master_units.parquet (which is definitely right), with the condo_listings, which is definitely not right
df['master_tmk'] = df['master_tmk'].astype('int64')
df['parcel_number'] = df['parcel_number'].astype('int64')
df['modified_address'] = df['modified_address'].astype('str')

df_1 = df.merge(master_address, left_on= 'parcel_number', right_on = 'parid', how = 'inner')

print(f"Missing rows from CCH data: {df.shape[0] - df_1.shape[0]}") # not too bad, but the new source of truth is better.

# we join the master parquet with the condo listings by using the levenstein distance between the two addresses as the way out

print(df["modified_address"].dtype)
print(condo_listings["address"].dtype)

print(df.loc[df["modified_address"].isna(), "modified_address"].head())
print(condo_listings.loc[condo_listings["address"].isna(), "address"].head())

print(df["modified_address"].map(type).value_counts())
print(condo_listings["address"].map(type).value_counts())

# to make it easier
import re
def slugify_address(text) -> str | None:
    if pd.isna(text):
        return None

    text = str(text).lower().strip()

    replacements = {
        r"\bstreet\b": "st",
        r"\broad\b": "rd",
        r"\bavenue\b": "ave",
        r"\bboulevard\b": "blvd",
        r"\bdrive\b": "dr",
        r"\blane\b": "ln",
        r"\bcourt\b": "ct",
        r"\bplace\b": "pl",
        r"\bparkway\b": "pkwy",
        r"\bnorth\b": "n",
        r"\bsouth\b": "s",
        r"\beast\b": "e",
        r"\bwest\b": "w",
        r"\bapartment\b": "apt",
        r"\bunit\b": "unit",
        r"\bnumber\b": "",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)

    text = re.sub(r"[^a-z0-9\s-]", "", text)
    text = re.sub(r"\s+", "-", text)
    text = re.sub(r"-+", "-", text)

    return text.strip("-")

df['modified_address'] = df['modified_address'].apply(slugify_address)
condo_listings['address'] = condo_listings['address'].apply(slugify_address)


print(df['modified_address'].head())
print(condo_listings['address'].head())

from levenpandas import fuzzymerge

merged = fuzzymerge(df, condo_listings, left_on='modified_address', right_on='address',
                    threshold=0.9, how='inner')   

print(merged)


Missing rows from CCH data: 437
str
str
228    NaN
656    NaN
657    NaN
658    NaN
659    NaN
Name: modified_address, dtype: str
Series([], Name: address, dtype: str)
modified_address
<class 'str'>      5493
<class 'float'>     331
Name: count, dtype: int64
address
<class 'str'>    597
Name: count, dtype: int64
0    66-queen-st-apt-1401
1    66-queen-st-apt-1402
2    66-queen-st-apt-1405
3    66-queen-st-apt-1501
4    66-queen-st-apt-1502
Name: modified_address, dtype: str
0    2380-kuhio-ave
1    2380-kuhio-ave
2    2380-kuhio-ave
3    2380-kuhio-ave
4    2380-kuhio-ave
Name: address, dtype: str
     master_tmk  parcel_number unit_number_x              modified_address  \
0  230390190000   230390190036           14B   1341-kapiolani-blvd-apt-14b   
1  230390190000   230390190043           16A   1341-kapiolani-blvd-apt-16a   
2  260100070000   260100070241          1209  1777-ala-moana-blvd-apt-1209   
3  260100070000   260100070293          1339  1777-ala-moana-blvd-apt-1339   
4  26

In [ ]:
# we now have to iterate on the other things and then do a close approximation of the data
# for some reason there is an import error for qpublic_unit but it works regardless

os.chdir('/home/kevinphan981/Programming/apartment-rental-scraper/src/qpublic')
os.getcwd()
os.listdir() # does contain qpublic_unit.py
from qpublic_unit import fetch_and_parse_parcel, json_to_dataframe

example_parcel = "260200690004"

record = fetch_and_parse_parcel(example_parcel)

record_df = json_to_dataframe(record)
print(record_df.head())


In [ ]:
# the issue here is that I have to go through every single one, which is not efficient
# I would rather use the subset of units that I have, roughly join them by address, and then use the parcel #s from master_tmk_address.csv to do it

final_df = pd.DataFrame()

for parcel in df['parcel_number'][:3]:
    record = fetch_and_parse_parcel(parcel)
    record_df = json_to_dataframe(record)
    pd.concat([final_df, record_df], ignore_index=True)
    print(f"Completed {parcel}")

print(final_df.head())